# Session 2: momentum, impulse and energy of a stop
**A. Real-world problem.** Session 1 gave F = m v^2 / (2 d) from energy. There is a second,
independent way to see a stop: *momentum and time*. A crash lasts a fraction of a second, and
the same stop can be described by how long it takes. Today you connect both views and then
solve a design problem: how much crumple distance keeps the driver's deceleration below
20 g from 50 km/h? And what changes at 100 km/h?

**Plan.** (1) momentum and impulse, (2) work and energy, (3) show the two views agree,
(4) a = v^2 / (2 d) and g-force, (5) design distance for a g limit, (6) the square law.

*Simplified educational model. The 20 g figure is only a commonly quoted rough scale for
brief loads; this lesson makes no injury predictions and is not a safety design tool.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.81  # m/s^2


def kmh_to_ms(v_kmh):
    return v_kmh / 3.6

## Step 1: momentum and impulse
Momentum is p = m v (kg m/s). Newton's second law says F = dp/dt, so over a short time the
*impulse* F*dt equals the change of momentum. For a constant force F lasting a time t:

    F * t = m * v          (the car goes from m v to 0)

In [ ]:
def momentum(m, v):
    return m * v


def force_from_time(m, v, t):
    """Constant force (N) that stops mass m (kg) at speed v (m/s) in time t (s)."""
    return m * v / t

## Step 2: work and energy
Work = force x distance. For a constant force F over a distance d: W = F*d. The work-energy
theorem says W equals the loss of kinetic energy, (1/2) m v^2. So F = m v^2 / (2 d).

In [ ]:
def kinetic_energy(m, v):
    return 0.5 * m * v * v


def force_from_distance(m, v, d):
    return kinetic_energy(m, v) / d

## Step 3: the two views agree
With constant deceleration a the average speed during the stop is v/2, so d = (v/2) t, i.e.
t = 2 d / v. Put that time into F = m v / t and you get F = m v^2 / (2 d) again.
Deceleration is a = F / m = v^2 / (2 d), and in "g" units it is a / g.

In [ ]:
def stop_time(v, d):
    """Duration (s) of a constant-deceleration stop from speed v (m/s) in distance d (m)."""
    return 2.0 * d / v


def avg_decel(v, d):
    """Average deceleration (m/s^2) = v^2 / (2 d). Independent of the mass."""
    return v * v / (2.0 * d)


def g_force(a):
    """Express an acceleration (m/s^2) as a multiple of g."""
    return a / G

## Step 4: the design problem (inverse use of the formula)
Choose the smallest crumple distance so the average deceleration stays at or below a limit
of `g_limit` g. Rearranging a = v^2/(2 d):  d = v^2 / (2 * g_limit * g).
Because v is squared, doubling the speed needs *four times* the distance for the same g.

In [ ]:
def distance_for_g(v, g_limit):
    return v * v / (2.0 * g_limit * G)


def max_speed_for_g(d, g_limit):
    """Highest speed (m/s) that a crumple distance d (m) can handle within g_limit."""
    return float(np.sqrt(2.0 * g_limit * G * d))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m = 1000.0
    for kmh in (50.0, 100.0):
        v = kmh_to_ms(kmh)
        d = distance_for_g(v, 20.0)
        print(f"{kmh:5.0f} km/h = {v:5.2f} m/s: KE {kinetic_energy(m, v) / 1000:6.1f} kJ, "
              f"distance for 20 g = {d:5.3f} m, stop time {stop_time(v, d) * 1000:5.1f} ms")
    v50 = kmh_to_ms(50.0)
    print(f"check, impulse view: F = m v / t = {force_from_time(m, v50, stop_time(v50, 0.5)) / 1000:.1f} kN; "
          f"energy view: {force_from_distance(m, v50, 0.5) / 1000:.1f} kN")
    print(f"ratio of distances 100 km/h to 50 km/h: {distance_for_g(kmh_to_ms(100), 20) / distance_for_g(v50, 20):.2f}")

    # Figure 1: deceleration (in g) against crumple distance for 50 and 100 km/h
    dist = np.linspace(0.2, 2.5, 200)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for kmh, col in ((50.0, "C0"), (100.0, "C3")):
        ax.plot(dist, g_force(avg_decel(kmh_to_ms(kmh), dist)), color=col, label=f"{kmh:.0f} km/h")
        dd = distance_for_g(kmh_to_ms(kmh), 20.0)
        ax.plot(dd, 20, "ko")
        ax.annotate(f"{dd:.2f} m", (dd, 20), textcoords="offset points", xytext=(6, 8), fontsize=8)
    ax.axhline(20, color="gray", ls="--", label="20 g (rough scale)")
    ax.set_ylim(0, 120)
    ax.set_xlabel("crumple distance d (m)")
    ax.set_ylabel("average deceleration (g)")
    ax.set_title("Deceleration against crumple distance")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "decel_vs_distance.png", dpi=120)

    # Figure 2: speed and position during a constant-deceleration stop from 50 km/h in 0.49 m
    d = distance_for_g(v50, 20.0)
    t = np.linspace(0, stop_time(v50, d), 100)
    a = avg_decel(v50, d)
    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6))
    a1.plot(t * 1000, v50 - a * t)
    a1.set_xlabel("time (ms)")
    a1.set_ylabel("car speed (m/s)")
    a1.set_title("Speed falls in a straight line")
    a1.grid(alpha=0.3)
    a2.plot(t * 1000, v50 * t - 0.5 * a * t ** 2)
    a2.set_xlabel("time (ms)")
    a2.set_ylabel("distance crushed (m)")
    a2.set_title(f"Reaches d = {d:.2f} m when it stops")
    a2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "stop_history.png", dpi=120)

## Coding tasks
1. **Easy.** Use `distance_for_g` to print the crumple distance for 20 g at 30, 50, 70, 100 km/h and check that the ratios follow the square of the speeds.
2. **Stretch.** Show in code that `force_from_time` and `force_from_distance` agree for three different (mass, speed, distance) sets, and explain why they must.
3. **Explain.** Explain why a 1000 kg and a 2000 kg car with the same crumple distance and speed have the same g but different forces and different kinetic energies.

## Assessment
- **A. Conceptual.** Why does doubling the speed need four times the crumple distance for the same deceleration, rather than twice?
- **B. Numerical.** A 1000 kg car hits at 80 km/h. What crumple distance keeps the average deceleration at or below 20 g? What speed can a 0.6 m crumple zone handle at 20 g?
- **C. Graph and simulation interpretation.** On the graph, a 1 m crumple distance at 50 km/h and at 100 km/h gives what g-forces? Which stays below the 20 g line?
- **D. Design and optimisation challenge.** Design: a car must meet 20 g at 64 km/h but the crumple zone is limited to 0.6 m. Is that possible? If not, what is the least distance, and what are the options (lower speed, higher g, more distance)? State the trade-offs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("momentum of 1000 kg at 10 m/s is 10000 kg m/s", abs(L.momentum(1000.0, 10.0) - 1.0e4) < 1e-9))
    checks.append(("hand calculation: stopping 10000 kg m/s in 0.1 s needs 100 kN",
                   abs(L.force_from_time(1000.0, 10.0, 0.1) - 1.0e5) < 1e-6))
    v = L.kmh_to_ms(50.0)
    f_t = L.force_from_time(1000.0, v, L.stop_time(v, 0.5))
    f_d = L.force_from_distance(1000.0, v, 0.5)
    checks.append(("impulse view and energy view give the same force", abs(f_t - f_d) < 1e-6))
    checks.append(("a = v^2 / (2 d) equals F / m", abs(L.avg_decel(v, 0.5) - f_d / 1000.0) < 1e-9))
    d50 = L.distance_for_g(v, 20.0)
    checks.append((f"worked example: 50 km/h and 20 g need d = {d50:.3f} m (0.492 m)", abs(d50 - 0.4917) < 5e-4))
    checks.append(("with that distance the deceleration is exactly 20 g",
                   abs(L.g_force(L.avg_decel(v, d50)) - 20.0) < 1e-9))
    d100 = L.distance_for_g(L.kmh_to_ms(100.0), 20.0)
    checks.append((f"square law: 100 km/h needs 4 times the distance ({d100:.3f} m)", abs(d100 / d50 - 4.0) < 1e-9))
    checks.append(("max_speed_for_g inverts distance_for_g",
                   abs(L.max_speed_for_g(d50, 20.0) - v) < 1e-9))
    checks.append(("stop time for 50 km/h in 0.4917 m is about 70.8 ms",
                   abs(L.stop_time(v, d50) - 0.0708) < 5e-4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
